<a href="https://colab.research.google.com/github/Vanquan99/Tool_Copy-Google-Drive/blob/main/tool_coppy_driver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Copy Folder / File — Google Drive to Google Drive
Dán link nguồn (`source_link`), link thư mục đích (`dest_link`) và bấm **Run** 1 lần duy nhất.

In [ ]:
#@title Copy Google Drive (Điền thông tin & bấm Run 1 lần)
source_link = "" #@param {type:"string"}
dest_link = "" #@param {type:"string"}
from_page = 0 #@param {type:"integer"}
to_page = 0 #@param {type:"integer"}
max_download_size_gb = 700 #@param {type:"number"}
exclude_str = "" #@param {type:"string"}
use_progress = True #@param {type:"boolean"}

import re, time
from google.colab import auth
from googleapiclient.discovery import build
from tqdm.notebook import tqdm

def extract_id(val):
    m = re.search(r'[-\w]{25,}', val or '')
    return m.group(0) if m else val.strip()

def esc(s):
    return s.replace('\\', '\\\\').replace("'", "\\'")

FOLDER_MIME = 'application/vnd.google-apps.folder'
src_id, dst_id = extract_id(source_link), extract_id(dest_link)
if not src_id or not dst_id:
    raise ValueError('Vui lòng nhập đầy đủ source_link và dest_link.')

auth.authenticate_user()
service = build('drive', 'v3')
excluded = [x.strip() for x in exclude_str.split(',') if x.strip()]
limit_mb = float(max_download_size_gb) * 1024 if max_download_size_gb > 0 else float('inf')
total_mb = 0.0

def log(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line)
    with open('copy_log.txt', 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def list_dest_map(folder_id):
    # shortcut: 1 lần list lấy toàn bộ mục hiện có trong folder đích thay vì gọi API cho từng file
    items, token = {}, None
    while True:
        res = service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields='files(id, name, mimeType), nextPageToken',
            pageSize=1000, pageToken=token,
            supportsAllDrives=True, includeItemsFromAllDrives=True
        ).execute()
        for f in res.get('files', []):
            items[f['name']] = f
        token = res.get('nextPageToken')
        if not token:
            break
    return items

def get_or_create_folder(parent_id, name, dest_map=None):
    existing = (dest_map or {}).get(name)
    if existing and existing.get('mimeType') == FOLDER_MIME:
        return existing['id']
    if dest_map is None:
        res = service.files().list(
            q=f"'{parent_id}' in parents and name = '{esc(name)}' and mimeType = '{FOLDER_MIME}' and trashed = false",
            fields='files(id)', supportsAllDrives=True, includeItemsFromAllDrives=True
        ).execute().get('files', [])
        if res:
            return res[0]['id']
    return service.files().create(
        body={'name': name, 'mimeType': FOLDER_MIME, 'parents': [parent_id]},
        fields='id', supportsAllDrives=True
    ).execute()['id']

def get_children(folder_id, fp=0, tp=0):
    files, token, page = [], None, 0
    q = f"'{folder_id}' in parents and trashed = false"
    if excluded:
        q += ' and (' + ' and '.join(f"not name contains '{esc(x)}'" for x in excluded) + ')'
    while True:
        res = service.files().list(
            q=q, orderBy='name, createdTime',
            fields='files(id, name, mimeType, size), nextPageToken',
            pageSize=1000, pageToken=token,
            supportsAllDrives=True, includeItemsFromAllDrives=True
        ).execute()
        page += 1
        if (fp < page <= tp) or tp == 0:
            files.extend(res.get('files', []))
        token = res.get('nextPageToken')
        if not token or (tp > 0 and page >= tp):
            break
    return files

def copy_file(parent_id, meta, dest_map):
    global total_mb
    if meta['name'] in dest_map:
        log(f"{meta['name']} - Đã tồn tại, bỏ qua")
        return True
    size_mb = int(meta.get('size', 0)) / (1024 * 1024) if meta.get('size') else 0.0
    if total_mb + size_mb > limit_mb:
        log(f"Đạt giới hạn dung lượng ({max_download_size_gb} GB), dừng sao chép.")
        return False
    t0 = time.time()
    service.files().copy(
        fileId=meta['id'], body={'parents': [parent_id]}, supportsAllDrives=True
    ).execute()
    total_mb += size_mb
    log(f"[{meta['name']}] Đã sao chép {size_mb:.2f} MB ({time.time() - t0:.2f}s) | Tổng: {total_mb / 1024:.2f} GB")
    return True

def copy_folder_contents(dest_folder_id, items):
    dest_map = list_dest_map(dest_folder_id)
    it = tqdm(items, desc='Đang sao chép') if use_progress else items
    for item in it:
        if item.get('mimeType') == FOLDER_MIME:
            log(f"Vào thư mục: {item['name']}")
            sub_id = get_or_create_folder(dest_folder_id, item['name'], dest_map)
            if not copy_folder_contents(sub_id, get_children(item['id'])):
                return False
        elif not copy_file(dest_folder_id, item, dest_map):
            return False
    return True

log('Bắt đầu sao chép')
src_meta = service.files().get(fileId=src_id, supportsAllDrives=True, fields='id,name,mimeType,size').execute()
if src_meta.get('mimeType') != FOLDER_MIME:
    copy_file(dst_id, src_meta, list_dest_map(dst_id))
else:
    target_id = get_or_create_folder(dst_id, src_meta['name'])
    copy_folder_contents(target_id, get_children(src_id, from_page, to_page))
log(f"Hoàn tất quá trình. Tổng dung lượng đã copy: {total_mb / 1024:.2f} GB")